### 평가지표 MAE MSE RMSE R^2

In [ ]:
# MAE
# 모든 오차에 동일한 가중치를 적용
# 이상치에 덜 민삼한 지표가 필요할 때, 오차를 직관적으로 파악하기 쉽다

In [ ]:
# MSE
# 오차를 제곱 : 1 미만의 오차는 작아지고 1 이상의 오차는 매우 커짐
# 모델이 오차가 큰 예측에 대해 큰 패널티를 부여 - 주로 사용
# 이상치를 제거하는게 아니라 이상치가 중요한 도메인 : 금융 리스크, 의료 진단

In [ ]:
# RMSE
# MSE에 제곱을 했으니 원래 데이터의 SCALE이 변함
# MSE의 장점을 그대로 가져가면서 MAE 장점도 가져감(오차가 큰 가중치에 패널티를 더 크게하면서 오차를 직관적으로 파악가능)

In [ ]:
# R^2
# 데이터의 분산을 설명하는 상대적인 성능 ( 0 ~ 1 )
# 객관적인 성능비교

# SSE (Sum of Squred Errors) : 오차 제곱의 합
# SST (Total Sum of Squares) : 실제값 들의 평균값(y_hat)과 실제값 들의 차이 제곱합


In [25]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

iris = load_iris(as_frame = True)
data = iris.data
X = data.iloc[:,:-1].to_numpy()
y = data['petal width (cm)'].to_numpy()
X, y

# 머신러닝
# 딥러닝
    # 평가지표를 다양하게
x_train,x_test,y_train,y_test = train_test_split( X , y, test_size=0.2, random_state=42 ) 

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor())
])

pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

print(mean_squared_error(y_test, y_pred))
print(mean_absolute_error(y_test, y_pred))
print(r2_score(y_test, y_pred))



0.04501143796296278
0.16463888888888842
0.9291889489998663


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader


torch.manual_seed(42)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train_t = torch.FloatTensor(x_train)
y_train_t = torch.FloatTensor(y_train)
x_test_t = torch.FloatTensor(x_test)
y_test_t = torch.FloatTensor(y_test)

train_dataset = TensorDataset(x_train_t, y_train_t)
test_dataset = TensorDataset(x_test_t, y_test_t)
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [71]:
class Irismodel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


In [ ]:
from tqdm import tqdm
from torch.optim import lr_scheduler

model = Irismodel(x_train_t.shape[-1])
criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.001)
scheduler = lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.1) # 이게 정확히 뭔 기능인데?
model.train()  # 이거 왜 하는거야

epochs = 100
for epoch in tqdm(range(epochs)):
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        y_pred = model(batch_x).squeeze(1)
        loss = criterion(y_pred, batch_y)
        loss.backward()
        optimizer.step()
        scheduler.step()

model.eval()
with torch.no_grad():
    test_pred = model(x_test_t)
    print(r2_score(y_test_t, test_pred))

100%|██████████| 100/100 [00:00<00:00, 129.67it/s]

0.8941344022750854


In [13]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

x_train,x_test,y_train,y_test = train_test_split( X , y, test_size=0.2, random_state=42 ) 
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [15]:
from xgboost import XGBRegressor
xgb = XGBRegressor()
xgb.fit(x_train, y_train)
r2_score(y_test, xgb.predict(x_test))

0.9150999096630922